# Final Test Comparison: 2.5D V4 vs 3D V2

Evaluates the canonical-grid 2.5D V4 model against the existing 3D V2 baseline on identical held-out private dataset pairs. V4 volumes are first resampled to `[96, 112, 96]`, all orientations use the strict `[B,112,96,16]` contract, and fused flows are mapped back to the original volume before image/segmentation metrics are calculated.

Registration quality includes anatomical-label-centroid TRE. For every shared private dataset segmentation label, the notebook measures the Euclidean distance in millimetres between the centroid of the warped moving label and the corresponding fixed label. This is a segmentation-derived centroid TRE, not manually annotated landmark TRE.

The notebook also writes board-parity local artifacts: per-device/per-pair V4 summaries, a matching quality chart, and a single-slice V4 preview. These use the same raw-space measurement pipeline as `fpga_inference_v4.ipynb`; local energy is explicitly unavailable rather than treated as FPGA telemetry.


In [ ]:
import os
from pathlib import Path
import json, os, platform, time
import numpy as np
import torch
import torch.nn.functional as F

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'Voxelmorph').exists():
    REPO_ROOT = REPO_ROOT.parent
VOX_DIR = REPO_ROOT / 'Voxelmorph'
DATA_ROOT = Path(os.environ.get('REGISTRATION_DATA_ROOT', str(REPO_ROOT / 'Data' / 'registration_dataset')))
V4_WEIGHTS = VOX_DIR / 'trained_weights' / '2p5d_dense_pt_v4_canonical_best.pth'
V3D_WEIGHTS = VOX_DIR / 'trained_weights' / '3d_dense_v2_best.pth'
OUTPUT_ROOT = VOX_DIR / 'artifacts' / 'results' / 'compare_2p5d_v4_fusion_3d_v2_test'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PAIR_SEED = 46
PAIR_LIMIT = None  # smoke test; set None for all held-out pairs after validation
REQUESTED_DEVICE_NAMES = ['cuda'] if torch.cuda.is_available() else ['cpu']
EVAL_BATCH_SIZE_V4 = 16  # batching changes local throughput only, not the V4 measurement pipeline or quality values
SMOOTH_SIGMA = 0.75
PIPELINE_VERSION = 'v4-canonical-letterbox'
LOCAL_PARITY_OUTPUT = OUTPUT_ROOT / 'benchmark_results_comparison_v4_local.json'
# Verified from private dataset NIfTI headers: seg_center and volumes_center share 1 mm isotropic geometry.
LABEL_CENTROID_TRE_SPACING_DHW_MM = (1.0, 1.0, 1.0)


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() not in {'0', 'false', 'no', 'off'}


# WSL can sample CUDA directly. CPU energy uses Linux RAPL when exposed, or
# LibreHardwareMonitor's web endpoint through the Windows/WSL localhost bridge.
# Set V4_MEASURE_LOCAL_POWER=0 to disable power collection explicitly.
IS_WSL = 'microsoft' in platform.release().lower() or 'wsl' in platform.release().lower()
LOCAL_POWER_ENABLED = env_flag('V4_MEASURE_LOCAL_POWER', platform.system().lower() == 'windows' or IS_WSL)
LOCAL_POWER_SAMPLE_INTERVAL_S = float(os.environ.get('V4_POWER_SAMPLE_INTERVAL_S', '0.10'))
LOCAL_POWER_IDLE_SECONDS = float(os.environ.get('V4_POWER_IDLE_SECONDS', '3.0'))
# GPU V4 stages are shorter than one sensor period. Repeating only the measured
# stage gives stable samples; reported energy and timing are divided per run.
LOCAL_POWER_INFERENCE_REPETITIONS = int(os.environ.get('V4_POWER_INFERENCE_REPETITIONS', '25'))
LOCAL_POWER_POSTPROCESS_REPETITIONS = int(os.environ.get('V4_POWER_POSTPROCESS_REPETITIONS', '1'))

assert V4_WEIGHTS.exists(), V4_WEIGHTS
assert V3D_WEIGHTS.exists(), V3D_WEIGHTS
print('Devices:', REQUESTED_DEVICE_NAMES)
print('Pairs:', PAIR_LIMIT or 'all')
print('V4 weights:', V4_WEIGHTS)
print('Local power:', 'enabled' if LOCAL_POWER_ENABLED else 'disabled', f'(sample {LOCAL_POWER_SAMPLE_INTERVAL_S:.2f}s)')


In [ ]:
# Reuse only the established data, metric, flow-lifting, and 3D-V2 definitions.
# The V2 2.5D inference and all quantization cells are deliberately not used.
legacy_path = VOX_DIR / 'compare_2p5d_fusion_3d_v2_test.ipynb'
legacy_nb = json.loads(legacy_path.read_text(encoding='utf-8'))
legacy_ns = globals()
configured_data_root = str(DATA_ROOT)
configured_pair_limit = PAIR_LIMIT
configured_devices = list(REQUESTED_DEVICE_NAMES)
configured_output_root = OUTPUT_ROOT
exec(''.join(legacy_nb['cells'][1]['source']), legacy_ns)
DATA_ROOT = configured_data_root
PAIR_LIMIT = configured_pair_limit
REQUESTED_DEVICE_NAMES = configured_devices
OUTPUT_ROOT = configured_output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for cell_index in (2, 3, 4, 5):
    exec(''.join(legacy_nb['cells'][cell_index]['source']), legacy_ns)

# Restore V4-specific configuration after legacy helper setup.
DATA_ROOT = configured_data_root
PAIR_LIMIT = configured_pair_limit
REQUESTED_DEVICE_NAMES = configured_devices
OUTPUT_ROOT = configured_output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_3D_PATH = str(V3D_WEIGHTS)
TARGET_VOL_SHAPE_3D = (96, 112, 96)
test_ds = load_split('test', None)
test_pairs = build_eval_pairs(len(test_ds), max_pairs=PAIR_LIMIT, seed=PAIR_SEED)
print('Test pairs:', test_pairs)


In [ ]:
import sys
import subprocess
import threading
import urllib.request
for candidate in (REPO_ROOT, VOX_DIR, VOX_DIR / '2.5D'):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
from vxm_2p5d_v4 import ORIENTATIONS as V4_ORIENTATIONS, combine_stacks, extract_stack as v4_extract_stack, letterbox_stack, letterbox_plane, resample_volume
from vxm_2p5d_export import load_model_for_export as load_board_cpu_reference


_NVML_HANDLE = None
_NVML_INITIALIZED = False


def _cpu_sensor_score(name):
    value = str(name).lower()
    if 'cpu package' in value:
        return 100
    if 'package' in value and 'cpu' in value:
        return 90
    if value in {'package', 'cpu total'}:
        return 80
    if 'ppt' in value:
        return 75
    if 'cpu' in value and 'power' in value:
        return 60
    return 0


def lhm_web_urls():
    """Return local and WSL-gateway URLs for LibreHardwareMonitor."""
    urls = ['http://127.0.0.1:8085/data.json', 'http://localhost:8085/data.json']
    if IS_WSL:
        try:
            for line in Path('/proc/net/route').read_text().splitlines()[1:]:
                fields = line.split()
                if len(fields) >= 3 and fields[1] == '00000000':
                    gateway_hex = fields[2]
                    octets = [str(int(gateway_hex[index:index + 2], 16)) for index in range(6, -1, -2)]
                    urls.append(f"http://{'.'.join(octets)}:8085/data.json")
                    break
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def read_linux_rapl_energy_j():
    """Return package energy from Linux RAPL when WSL exposes it."""
    base = Path('/sys/class/powercap')
    if not base.exists():
        return None
    values = []
    for energy_path in sorted(base.glob('intel-rapl:*/energy_uj')):
        if energy_path.parent.name.count(':') != 1:
            continue
        try:
            values.append(float(energy_path.read_text().strip()) / 1e6)
        except Exception:
            pass
    return None if not values else float(sum(values))


def query_local_cpu_power_w():
    """Read CPU package power via Windows WMI or the WSL-accessible LHM web server."""
    if platform.system().lower() == 'windows':
        try:
            import wmi
            candidates = []
            for namespace in (r'root\LibreHardwareMonitor', r'root\OpenHardwareMonitor'):
                try:
                    sensors = wmi.WMI(namespace=namespace).Sensor()
                except Exception:
                    continue
                for sensor in sensors:
                    if str(getattr(sensor, 'SensorType', '')).lower() != 'power':
                        continue
                    score = _cpu_sensor_score(getattr(sensor, 'Name', ''))
                    value = getattr(sensor, 'Value', None)
                    if score and value is not None:
                        candidates.append((score, float(value)))
            if candidates:
                return max(candidates, key=lambda item: item[0])[1]
        except Exception:
            pass

    # LibreHardwareMonitor's remote web server is reachable from modern WSL
    # through localhost. It is the CPU-power fallback when RAPL is unavailable.
    for url in lhm_web_urls():
        try:
            with urllib.request.urlopen(url, timeout=1.0) as response:
                tree = json.loads(response.read().decode('utf-8', errors='replace'))
        except Exception:
            continue

        def walk(node, context=''):
            if not isinstance(node, dict):
                return []
            name = str(node.get('Text') or node.get('text') or node.get('Name') or node.get('name') or '')
            full_name = f'{context} {name}'.strip()
            value = node.get('Value', node.get('value'))
            found = []
            score = _cpu_sensor_score(full_name)
            if score and value is not None:
                try:
                    found.append((score, float(str(value).replace(',', '.').split()[0])))
                except Exception:
                    pass
            for key in ('Children', 'children'):
                for child in node.get(key, []) if isinstance(node.get(key), list) else []:
                    found.extend(walk(child, full_name))
            return found

        candidates = walk(tree)
        if candidates:
            return max(candidates, key=lambda item: item[0])[1]
    return None


def query_local_gpu_power_w():
    """Prefer NVML's in-process query; retain nvidia-smi as a safe fallback."""
    global _NVML_HANDLE, _NVML_INITIALIZED
    if not torch.cuda.is_available():
        return None
    if not _NVML_INITIALIZED:
        _NVML_INITIALIZED = True
        try:
            import pynvml
            pynvml.nvmlInit()
            _NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
        except Exception:
            _NVML_HANDLE = None
    if _NVML_HANDLE is not None:
        try:
            import pynvml
            return float(pynvml.nvmlDeviceGetPowerUsage(_NVML_HANDLE) / 1000.0)
        except Exception:
            pass
    try:
        process = subprocess.run(
            ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits', '-i', '0'],
            capture_output=True, text=True, timeout=2.0, check=False,
        )
        if process.returncode == 0 and process.stdout.strip():
            return float(process.stdout.strip().splitlines()[0])
    except Exception:
        pass
    return None


def _process_rss_mb():
    try:
        import psutil
        return float(psutil.Process().memory_info().rss / (1024 ** 2))
    except Exception:
        return None


class LocalPowerMonitor:
    """Local analogue of the FPGA rail monitor with the same energy formula."""
    def __init__(self, device_name, idle_gpu_w=0.0, idle_cpu_w=0.0, sample_interval_s=LOCAL_POWER_SAMPLE_INTERVAL_S):
        self.device_name = str(device_name)
        self.idle_gpu_w = float(idle_gpu_w or 0.0)
        self.idle_cpu_w = float(idle_cpu_w or 0.0)
        self.sample_interval_s = max(float(sample_interval_s), 0.02)
        self.gpu_samples_w, self.cpu_samples_w, self.memory_samples_mb = [], [], []
        self._stop = False
        self._thread = None
        self._start_wall = self._end_wall = None
        self._start_rss_mb = self._end_rss_mb = None
        self._start_cpu_rapl_j = self._end_cpu_rapl_j = None

    def _sample(self):
        sample_gpu = self.device_name == 'cuda'
        while not self._stop:
            if sample_gpu:
                value = query_local_gpu_power_w()
                if value is not None:
                    self.gpu_samples_w.append(float(value))
            value = query_local_cpu_power_w()
            if value is not None:
                self.cpu_samples_w.append(float(value))
            value = _process_rss_mb()
            if value is not None:
                self.memory_samples_mb.append(value)
            time.sleep(self.sample_interval_s)

    def __enter__(self):
        self._start_cpu_rapl_j = read_linux_rapl_energy_j()
        self._start_rss_mb = _process_rss_mb()
        if self._start_rss_mb is not None:
            self.memory_samples_mb.append(self._start_rss_mb)
        self._start_wall = time.perf_counter()
        if LOCAL_POWER_ENABLED:
            self._thread = threading.Thread(target=self._sample, daemon=True)
            self._thread.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self._end_wall = time.perf_counter()
        self._end_cpu_rapl_j = read_linux_rapl_energy_j()
        self._end_rss_mb = _process_rss_mb()
        if self._end_rss_mb is not None:
            self.memory_samples_mb.append(self._end_rss_mb)
        self._stop = True
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        return False

    def result(self):
        wall_time_s = max(float((self._end_wall or 0.0) - (self._start_wall or 0.0)), 0.0)
        gpu_mean = float(np.mean(self.gpu_samples_w)) if self.gpu_samples_w else None
        cpu_mean = float(np.mean(self.cpu_samples_w)) if self.cpu_samples_w else None
        gpu_energy = None if gpu_mean is None else float(gpu_mean * wall_time_s)
        cpu_energy = None if cpu_mean is None else float(cpu_mean * wall_time_s)
        cpu_backend = 'LibreHardwareMonitor/OpenHardwareMonitor' if self.cpu_samples_w else None
        # WSL generally does not expose Windows WMI. If Linux RAPL is available,
        # its energy counter provides an equivalent CPU-package energy reading.
        if cpu_energy is None and self._start_cpu_rapl_j is not None and self._end_cpu_rapl_j is not None:
            rapl_delta = float(self._end_cpu_rapl_j - self._start_cpu_rapl_j)
            if rapl_delta >= 0.0:
                cpu_energy = rapl_delta
                cpu_mean = None if wall_time_s <= 0 else float(cpu_energy / wall_time_s)
                cpu_backend = 'Linux RAPL'
        gpu_dynamic = None if gpu_energy is None else float(max(gpu_energy - self.idle_gpu_w * wall_time_s, 0.0))
        cpu_dynamic = None if cpu_energy is None else float(max(cpu_energy - self.idle_cpu_w * wall_time_s, 0.0))
        energy_parts = [value for value in (cpu_energy, gpu_energy) if value is not None]
        dynamic_parts = [value for value in (cpu_dynamic, gpu_dynamic) if value is not None]
        return {
            'power_wall_time_s': wall_time_s,
            'cpu_energy_j': None if cpu_energy is None else float(cpu_energy),
            'cpu_dynamic_energy_j': None if cpu_dynamic is None else float(cpu_dynamic),
            'cpu_power_mean_w': cpu_mean,
            'cpu_power_peak_w': None if not self.cpu_samples_w else float(np.max(self.cpu_samples_w)),
            'cpu_power_samples': int(len(self.cpu_samples_w)),
            'cpu_power_backend': cpu_backend,
            'gpu_energy_j': None if gpu_energy is None else float(gpu_energy),
            'gpu_dynamic_energy_j': None if gpu_dynamic is None else float(gpu_dynamic),
            'gpu_power_mean_w': gpu_mean,
            'gpu_power_peak_w': None if not self.gpu_samples_w else float(np.max(self.gpu_samples_w)),
            'gpu_power_samples': int(len(self.gpu_samples_w)),
            'energy_j': None if not energy_parts else float(sum(energy_parts)),
            'dynamic_energy_j': None if not dynamic_parts else float(sum(dynamic_parts)),
            'power_mean_w': None if not energy_parts or wall_time_s <= 0 else float(sum(energy_parts) / wall_time_s),
            'process_rss_start_mb': self._start_rss_mb,
            'process_rss_end_mb': self._end_rss_mb,
            'process_rss_peak_mb': None if not self.memory_samples_mb else float(np.max(self.memory_samples_mb)),
            'process_rss_delta_mb': None if self._start_rss_mb is None or self._end_rss_mb is None else float(self._end_rss_mb - self._start_rss_mb),
            'process_memory_samples': int(len(self.memory_samples_mb)),
        }


def _empty_local_power():
    return {
        'power_wall_time_s': None, 'cpu_energy_j': None, 'cpu_dynamic_energy_j': None,
        'cpu_power_mean_w': None, 'cpu_power_peak_w': None, 'cpu_power_samples': 0, 'cpu_power_backend': None,
        'gpu_energy_j': None, 'gpu_dynamic_energy_j': None, 'gpu_power_mean_w': None,
        'gpu_power_peak_w': None, 'gpu_power_samples': 0, 'energy_j': None,
        'dynamic_energy_j': None, 'power_mean_w': None, 'process_rss_start_mb': None,
        'process_rss_end_mb': None, 'process_rss_peak_mb': None,
        'process_rss_delta_mb': None, 'process_memory_samples': 0,
    }


def calibrate_local_idle_power(runtime_device):
    """Match the board's three-second rail calibration after model warm-up."""
    if not LOCAL_POWER_ENABLED:
        return {'enabled': False, 'idle_seconds': LOCAL_POWER_IDLE_SECONDS, 'idle_gpu_w': None, 'idle_cpu_w': None}
    with LocalPowerMonitor(runtime_device.type) as meter:
        time.sleep(LOCAL_POWER_IDLE_SECONDS)
    result = meter.result()
    return {
        'enabled': True,
        'idle_seconds': LOCAL_POWER_IDLE_SECONDS,
        'idle_gpu_w': result['gpu_power_mean_w'],
        'idle_cpu_w': result['cpu_power_mean_w'],
        'gpu_samples': result['gpu_power_samples'],
        'cpu_samples': result['cpu_power_samples'],
    }


def measure_local_stage(stage, runtime_device, idle_power, repetitions=1, duration_getter=None):
    """Run a board-equivalent stage repeatedly and normalize energy per run."""
    repetitions = max(int(repetitions), 1)
    last_result, durations = None, []
    if LOCAL_POWER_ENABLED:
        meter = LocalPowerMonitor(runtime_device.type, idle_gpu_w=idle_power.get('idle_gpu_w'), idle_cpu_w=idle_power.get('idle_cpu_w'))
        with meter:
            for _ in range(repetitions):
                last_result = stage()
                if duration_getter is not None:
                    durations.append(float(duration_getter(last_result)))
        power = meter.result()
        for key in ('power_wall_time_s', 'cpu_energy_j', 'cpu_dynamic_energy_j', 'gpu_energy_j', 'gpu_dynamic_energy_j', 'energy_j', 'dynamic_energy_j'):
            if power.get(key) is not None:
                power[key] = float(power[key] / repetitions)
        # Means and peaks are physical readings; only per-run energy/time is normalized.
        power['power_repetitions'] = repetitions
    else:
        last_result = stage()
        if duration_getter is not None:
            durations.append(float(duration_getter(last_result)))
        power = _empty_local_power()
        power['power_repetitions'] = 1
    mean_duration = None if not durations else float(np.mean(durations))
    return last_result, power, mean_duration


def combine_local_power_measurements(parts):
    parts = [part for part in parts if part]
    if not parts:
        return _empty_local_power()

    def sum_known(key):
        values = [part.get(key) for part in parts if part.get(key) is not None]
        return None if not values else float(sum(values))

    def max_known(key):
        values = [part.get(key) for part in parts if part.get(key) is not None]
        return None if not values else float(max(values))

    wall_time_s = sum_known('power_wall_time_s')
    cpu_energy = sum_known('cpu_energy_j')
    gpu_energy = sum_known('gpu_energy_j')
    energy = sum_known('energy_j')
    return {
        'power_wall_time_s': wall_time_s,
        'cpu_energy_j': cpu_energy,
        'cpu_dynamic_energy_j': sum_known('cpu_dynamic_energy_j'),
        'cpu_power_mean_w': None if cpu_energy is None or not wall_time_s else float(cpu_energy / wall_time_s),
        'cpu_power_peak_w': max_known('cpu_power_peak_w'),
        'cpu_power_samples': int(sum(part.get('cpu_power_samples', 0) for part in parts)),
        'cpu_power_backend': next((part.get('cpu_power_backend') for part in parts if part.get('cpu_power_backend')), None),
        'gpu_energy_j': gpu_energy,
        'gpu_dynamic_energy_j': sum_known('gpu_dynamic_energy_j'),
        'gpu_power_mean_w': None if gpu_energy is None or not wall_time_s else float(gpu_energy / wall_time_s),
        'gpu_power_peak_w': max_known('gpu_power_peak_w'),
        'gpu_power_samples': int(sum(part.get('gpu_power_samples', 0) for part in parts)),
        'energy_j': energy,
        'dynamic_energy_j': sum_known('dynamic_energy_j'),
        'power_mean_w': None if energy is None or not wall_time_s else float(energy / wall_time_s),
        'process_rss_start_mb': parts[0].get('process_rss_start_mb'),
        'process_rss_end_mb': parts[-1].get('process_rss_end_mb'),
        'process_rss_peak_mb': max_known('process_rss_peak_mb'),
        'process_rss_delta_mb': sum_known('process_rss_delta_mb'),
        'process_memory_samples': int(sum(part.get('process_memory_samples', 0) for part in parts)),
        'power_repetitions': max(int(part.get('power_repetitions', 1)) for part in parts),
    }


def canonical_pair(dataset, moving_idx, fixed_idx):
    moving, _, moving_seg, _ = get_sample_parts(dataset, moving_idx)
    _, fixed, _, fixed_seg = get_sample_parts(dataset, fixed_idx)
    moving_c = normalize_volume_contract(resample_volume(moving))
    fixed_c = normalize_volume_contract(resample_volume(fixed))
    moving_seg_c = resample_volume(moving_seg, is_segmentation=True)
    fixed_seg_c = resample_volume(fixed_seg, is_segmentation=True)
    return moving, fixed, moving_seg, fixed_seg, moving_c, fixed_c, moving_seg_c, fixed_seg_c


def canvas_flow_to_native(flow, orientation):
    if orientation == 'axial':
        return flow
    if orientation == 'coronal':
        return flow[:, :, 8:104, :]
    native = np.empty((flow.shape[0], 2, 96, 112), dtype=np.float32)
    native[:, 0] = flow[:, 1].transpose(0, 2, 1)  # H displacement
    native[:, 1] = flow[:, 0].transpose(0, 2, 1)  # D displacement
    return native


@torch.no_grad()
def infer_v4_orientation(model, moving, fixed, orientation, runtime_device):
    axis = int(V4_ORIENTATIONS[orientation]['axis'])
    indices = list(range(3, moving.shape[axis] - 3))
    flows, elapsed = [], 0.0
    for start in range(0, len(indices), EVAL_BATCH_SIZE_V4):
        samples = []
        for z in indices[start:start + EVAL_BATCH_SIZE_V4]:
            m_stack, _ = letterbox_stack(v4_extract_stack(moving, orientation, z), orientation)
            f_stack, _ = letterbox_stack(v4_extract_stack(fixed, orientation, z), orientation)
            samples.append(combine_stacks(m_stack, f_stack))
        batch = torch.from_numpy(np.stack(samples)).permute(0, 2, 3, 1).to(runtime_device)
        if runtime_device.type == 'cuda':
            torch.cuda.synchronize(runtime_device)
        t0 = time.perf_counter(); flow = model(batch)
        if runtime_device.type == 'cuda':
            torch.cuda.synchronize(runtime_device)
        elapsed += (time.perf_counter() - t0) * 1000.0
        flows.append(flow.cpu().numpy())
    return canvas_flow_to_native(np.concatenate(flows), orientation), elapsed


def canonical_field_to_raw(field, raw_shape):
    out = np.empty((3, *raw_shape), dtype=np.float32)
    scale = (raw_shape[2] / 96.0, raw_shape[1] / 112.0, raw_shape[0] / 96.0)
    for channel, factor in enumerate(scale):
        out[channel] = resize_component_3d(field[channel], raw_shape) * factor
    return out


def warm_v4_model(model, canonical, runtime_device):
    moving_c, fixed_c = canonical[0], canonical[1]
    for orientation in ('axial', 'coronal', 'sagittal'):
        infer_v4_orientation(model, moving_c, fixed_c, orientation, runtime_device)


def run_v4_pair(model, raw, canonical, runtime_device, idle_power):
    moving, fixed, moving_seg, fixed_seg = raw
    moving_c, fixed_c, _, _ = canonical
    inference_repetitions = LOCAL_POWER_INFERENCE_REPETITIONS if runtime_device.type == 'cuda' else 1

    def run_orientation(orientation, lifter):
        def stage():
            flow, elapsed = infer_v4_orientation(model, moving_c, fixed_c, orientation, runtime_device)
            return lifter(flow, moving_c.shape), elapsed
        result, power, elapsed = measure_local_stage(stage, runtime_device, idle_power, inference_repetitions, lambda item: item[1])
        return result[0], float(elapsed), power

    field_a, t_ax, axial_power = run_orientation('axial', lift_axial)
    field_c, t_co, coronal_power = run_orientation('coronal', lift_coronal)
    field_s, t_sa, sagittal_power = run_orientation('sagittal', lift_sagittal)
    mean_f = fuse_fields(field_a, field_c, field_s)
    variants = {'v4_axial': field_a, 'v4_mean_fused': mean_f, 'v4_smoothed_0p75': smooth_field_3d(mean_f, SMOOTH_SIGMA)}
    inference_power_parts = {
        'v4_axial': [axial_power],
        'v4_mean_fused': [axial_power, coronal_power, sagittal_power],
        'v4_smoothed_0p75': [axial_power, coronal_power, sagittal_power],
    }
    output = {}
    for name, canonical_flow in variants.items():
        def post_stage():
            t0 = time.perf_counter()
            raw_flow = canonical_field_to_raw(canonical_flow, moving.shape)
            warped = warp_volume_3d(moving, raw_flow, mode='bilinear', runtime_device=runtime_device)
            warped_seg = warp_volume_3d(moving_seg.astype(np.float32), raw_flow, mode='nearest', runtime_device=runtime_device).astype(np.int16)
            metrics = summarize_registration(moving, fixed, moving_seg, fixed_seg, raw_flow, warped, warped_seg)
            metrics.update(label_centroid_tre_metrics(moving_seg, warped_seg, fixed_seg))
            return metrics, raw_flow, (time.perf_counter() - t0) * 1000.0

        result, post_power, post_ms = measure_local_stage(
            post_stage, runtime_device, idle_power, LOCAL_POWER_POSTPROCESS_REPETITIONS, lambda item: item[2],
        )
        metrics, raw_flow, _ = result
        metrics.update(
            model_inference_ms=t_ax if name == 'v4_axial' else t_ax + t_co + t_sa,
            postprocess_ms=float(post_ms),
        )
        metrics['total_runtime_ms'] = metrics['model_inference_ms'] + metrics['postprocess_ms']
        metrics.update(combine_local_power_measurements(inference_power_parts[name] + [post_power]))
        output[name] = (metrics, raw_flow)
    return output


In [ ]:
# Anatomical-label-centroid TRE (segmentation-derived)
#
# The VoxelMorph transform is a pull warp: `warped_seg` already lives in fixed-image
# coordinates. Comparing centroids of warped moving labels against fixed labels therefore
# avoids treating the backward flow as a forward landmark displacement.
#
# Labels are the same held-out anatomical labels used by the Dice metric. The masks are
# evaluation targets only and are never supplied to either model at inference time.

LABEL_CENTROID_TRE_LABELS = tuple(SEG_LABELS)

def _label_centroid_dhw(segmentation, label):
    coordinates = np.argwhere(segmentation == label)
    if coordinates.size == 0:
        return None
    return coordinates.mean(axis=0).astype(np.float32)


def label_centroid_tre_metrics(moving_seg, warped_seg, fixed_seg, labels=LABEL_CENTROID_TRE_LABELS, spacing_dhw_mm=LABEL_CENTROID_TRE_SPACING_DHW_MM):
    """Return pre/post segmentation-derived centroid TRE values in millimetres."""
    spacing = np.asarray(spacing_dhw_mm, dtype=np.float32)
    if spacing.shape != (3,) or np.any(spacing <= 0):
        raise ValueError(f'Expected positive [D,H,W] spacing, got {spacing_dhw_mm}')

    before_errors, after_errors, per_label_after = [], [], {}
    eligible_labels, missing_warped_labels = [], []
    for label in labels:
        moving_center = _label_centroid_dhw(moving_seg, label)
        fixed_center = _label_centroid_dhw(fixed_seg, label)
        if moving_center is None or fixed_center is None:
            continue
        eligible_labels.append(int(label))
        before_errors.append(float(np.linalg.norm((moving_center - fixed_center) * spacing)))

        warped_center = _label_centroid_dhw(warped_seg, label)
        if warped_center is None:
            missing_warped_labels.append(int(label))
            continue
        error_mm = float(np.linalg.norm((warped_center - fixed_center) * spacing))
        after_errors.append(error_mm)
        per_label_after[str(int(label))] = error_mm

    result = {
        'tre_before_mm': float(np.mean(before_errors)) if before_errors else None,
        'tre_mm': float(np.mean(after_errors)) if after_errors else None,
        'tre_median_mm': float(np.median(after_errors)) if after_errors else None,
        'tre_label_count': int(len(after_errors)),
        'tre_eligible_label_count': int(len(eligible_labels)),
        'tre_missing_warped_labels': missing_warped_labels,
        'tre_per_label_mm': per_label_after,
    }
    return result


In [ ]:
rows = []
LOCAL_POWER_CALIBRATIONS = {}
for device_name in REQUESTED_DEVICE_NAMES:
    runtime_device = torch.device(device_name)
    # Same export-compatible wrapper used by fpga_inference_v4.ipynb. Its V4
    # output is numerically identical to Vxm2p5dV4 while accepting NHWC input.
    v4_model = load_board_cpu_reference(V4_WEIGHTS).to(runtime_device).eval()
    model_3d = VxmDense3DV2().to(runtime_device).eval()
    state_3d = torch.load(V3D_WEIGHTS, map_location='cpu', weights_only=False)
    model_3d.load_state_dict(state_3d.get('state_dict', state_3d))

    # Warm the model before the idle baseline, so CUDA initialization and cache
    # allocation are not counted as registration energy.
    if LOCAL_POWER_ENABLED and test_pairs:
        warm_values = canonical_pair(test_ds, *test_pairs[0])
        warm_v4_model(v4_model, warm_values[4:], runtime_device)
        if runtime_device.type == 'cuda':
            torch.cuda.synchronize(runtime_device)
    print(f'Running local power calibration for {device_name} ({LOCAL_POWER_IDLE_SECONDS:.1f}s idle)...')
    idle_power = calibrate_local_idle_power(runtime_device)
    LOCAL_POWER_CALIBRATIONS[device_name] = idle_power
    print('  Idle power:', idle_power)

    for pair_index, (moving_idx, fixed_idx) in enumerate(test_pairs):
        values = canonical_pair(test_ds, moving_idx, fixed_idx)
        raw, canonical = values[:4], values[4:]
        results = run_v4_pair(v4_model, raw, canonical, runtime_device, idle_power)
        summary_3d, _, warped_seg_3d, flow_3d = run_3d_pair(model_3d, *raw, runtime_device)
        summary_3d.update(label_centroid_tre_metrics(raw[2], warped_seg_3d, raw[3]))
        results['3d_v2'] = (summary_3d, flow_3d)
        for method, (metrics, flow) in results.items():
            metrics = dict(metrics)
            metrics.update(device=device_name, pair_index=pair_index, moving_idx=moving_idx, fixed_idx=fixed_idx, method=method)
            rows.append(metrics)
        print(device_name, pair_index, {name: round(item[0]['dice_after'], 4) for name, item in results.items()})

(OUTPUT_ROOT / 'v4_vs_3d_rows.json').write_text(json.dumps(rows, indent=2))
print('Wrote', OUTPUT_ROOT / 'v4_vs_3d_rows.json')


In [ ]:
from collections import defaultdict
summary = defaultdict(lambda: defaultdict(list))
for row in rows:
    for key in ('dice_after', 'mi_after', 'ssim_deformed_fixed', 'total_runtime_ms', 'tre_before_mm', 'tre_mm', 'tre_median_mm', 'tre_label_count'):
        if row.get(key) is not None:
            summary[row['method']][key].append(row[key])
for method, metrics in summary.items():
    print(method, {key: round(float(np.mean(values)), 5) for key, values in metrics.items()})
print('TRE is segmentation-derived: mean distance between warped-moving and fixed anatomical-label centroids (mm).')


In [ ]:
# Board-parity local artifacts
# The local implementation uses the same 3-second idle baseline and
# mean-power-times-duration energy accounting as fpga_inference_v4.ipynb.
from collections import defaultdict

V4_METHODS = ('v4_axial', 'v4_mean_fused', 'v4_smoothed_0p75')
rows_by_device = defaultdict(list)
for row in rows:
    rows_by_device[row['device']].append(row)


def _mean_or_none(values):
    values = [value for value in values if value is not None]
    return None if not values else float(np.mean(values))


def _max_or_none(values):
    values = [value for value in values if value is not None]
    return None if not values else float(np.max(values))


def _device_payload(device_rows):
    pairs = {}
    for row in device_rows:
        if row['method'] not in V4_METHODS:
            continue
        pair = pairs.setdefault(str(row['pair_index']), {
            'pair_index': int(row['pair_index']),
            'moving_idx': int(row['moving_idx']),
            'fixed_idx': int(row['fixed_idx']),
        })
        pair[row['method']] = {key: value for key, value in row.items() if key not in ('device', 'pair_index', 'moving_idx', 'fixed_idx', 'method')}

    methods = {}
    aggregate_keys = (
        'dice_before', 'dice_after', 'mi_before', 'mi_after',
        'ssim_deformed_fixed', 'ssim_deformed_moving', 'tre_before_mm',
        'tre_mm', 'tre_median_mm', 'tre_label_count', 'model_inference_ms',
        'postprocess_ms', 'total_runtime_ms', 'power_wall_time_s', 'energy_j',
        'dynamic_energy_j', 'cpu_energy_j', 'cpu_dynamic_energy_j',
        'cpu_power_mean_w', 'gpu_energy_j', 'gpu_dynamic_energy_j',
        'gpu_power_mean_w', 'power_mean_w', 'process_rss_delta_mb',
    )
    for method in V4_METHODS:
        records = [row for row in device_rows if row['method'] == method]
        if not records:
            continue
        methods[method] = {key: _mean_or_none([record.get(key) for record in records]) for key in aggregate_keys}
        methods[method].update({
            'cpu_power_peak_w': _max_or_none([record.get('cpu_power_peak_w') for record in records]),
            'gpu_power_peak_w': _max_or_none([record.get('gpu_power_peak_w') for record in records]),
            'cpu_power_samples': int(sum(record.get('cpu_power_samples', 0) for record in records)),
            'gpu_power_samples': int(sum(record.get('gpu_power_samples', 0) for record in records)),
            'process_rss_peak_mb': _max_or_none([record.get('process_rss_peak_mb') for record in records]),
            'power_repetitions': int(max(record.get('power_repetitions', 1) for record in records)),
        })
    return {'methods': methods, 'pairs': [pairs[key] for key in sorted(pairs, key=int)]}


local_devices = {device: _device_payload(device_rows) for device, device_rows in rows_by_device.items()}
three_d_rows = [row for row in rows if row['method'] == '3d_v2']
parity_payload = {
    'pipeline_version': PIPELINE_VERSION,
    'canonical_volume_shape': [96, 112, 96],
    'input_shape_nhwc': [1, 112, 96, 16],
    'tre_definition': 'segmentation-derived anatomical-label-centroid TRE in mm',
    'measurement_pipeline': 'same V4 raw-space Dice/MI/SSIM/TRE and per-orientation/postprocess power boundaries as fpga_inference_v4.ipynb; E=mean(power)*duration; dynamic energy subtracts the 3-second idle baseline',
    'measurement_environment': 'WSL/local: CUDA GPU via NVML (nvidia-smi fallback); CPU package via Linux RAPL when exposed or Libre/OpenHardwareMonitor web through localhost; values are null when sensors are unavailable',
    'calibration': {'local': LOCAL_POWER_CALIBRATIONS},
    'power_repetitions': {
        'inference_cuda': LOCAL_POWER_INFERENCE_REPETITIONS,
        'postprocess': LOCAL_POWER_POSTPROCESS_REPETITIONS,
        'note': 'Only short CUDA inference stages are repeated; energy and timing are normalized per logical run.',
    },
    'pair_count': len({row['pair_index'] for row in rows}),
    'device_order': list(local_devices),
    'additional_local_baseline': {'3d_v2_rows': three_d_rows},
}
# Match the board artifact hierarchy: each concrete device owns `methods` and `pairs`.
# CUDA is intentionally not renamed to FPGA.
parity_payload.update(local_devices)
LOCAL_PARITY_OUTPUT.write_text(json.dumps(parity_payload, indent=2))
print('Wrote', LOCAL_PARITY_OUTPUT)

active_devices = list(local_devices)
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
x = np.arange(len(V4_METHODS))
width = min(0.7 / max(len(active_devices), 1), 0.35)
for index, device in enumerate(active_devices):
    methods = local_devices[device]['methods']
    offset = (index - (len(active_devices) - 1) / 2) * width
    label = f'Local {device.upper()}'
    axes[0].bar(x + offset, [methods[method]['model_inference_ms'] for method in V4_METHODS], width, label=label)
    energy = [methods[method]['dynamic_energy_j'] for method in V4_METHODS]
    if all(value is not None for value in energy):
        axes[1].bar(x + offset, energy, width, label=label)
    axes[2].bar(x + offset, [methods[method]['dice_after'] for method in V4_METHODS], width, label=label)
    axes[3].bar(x + offset, [methods[method]['tre_mm'] for method in V4_METHODS], width, label=label)
first_methods = local_devices[active_devices[0]]['methods']
axes[2].bar(x - 0.38, [first_methods[method]['dice_before'] for method in V4_METHODS], min(width, 0.22), color='#9E9E9E', label='Before registration')
axes[3].bar(x - 0.38, [first_methods[method]['tre_before_mm'] for method in V4_METHODS], min(width, 0.22), color='#9E9E9E', label='Before registration')
if not any(methods[method]['dynamic_energy_j'] is not None for methods in (payload['methods'] for payload in local_devices.values()) for method in V4_METHODS):
    axes[1].text(0.5, 0.5, 'Dynamic energy unavailable\n(check LibreHardwareMonitor/NVML)', ha='center', va='center', transform=axes[1].transAxes)
for axis, title, ylabel in zip(axes, ('Inference latency', 'Dynamic energy', 'Dice accuracy', 'Label-centroid TRE'), ('Model ms (lower is better)', 'Joules per logical run (lower is better)', 'Dice (higher is better)', 'mm (lower is better)')):
    axis.set_title(title); axis.set_ylabel(ylabel); axis.set_xticks(x); axis.set_xticklabels([method.upper() for method in V4_METHODS], rotation=15); axis.grid(axis='y', linestyle='--', alpha=0.5)
axes[2].set_ylim(0, 1.0)
axes[0].legend(); axes[2].legend(); axes[3].legend()
if axes[1].get_legend_handles_labels()[0]:
    axes[1].legend()
plt.tight_layout()
chart_path = OUTPUT_ROOT / 'benchmark_charts_v4.png'
plt.savefig(chart_path, dpi=150)
if os.environ.get('V4_NONINTERACTIVE', '0') == '1':
    plt.close(fig)
else:
    plt.show()

summary_lines = [
    '# Local V4 Board-Parity Summary',
    '',
    'Quality, timing, idle-baseline subtraction, and per-stage power boundaries match `fpga_inference_v4.ipynb`. CPU package energy uses Linux RAPL when WSL exposes it, otherwise the Libre/OpenHardwareMonitor localhost web endpoint; CUDA power uses NVML with an `nvidia-smi` fallback. Short CUDA inference stages are repeated only for power sampling, then normalized per logical run.',
    '',
    '| Device | Method | Dice after | TRE (mm) | Dynamic energy (J) | Model ms | Post-process ms | Total ms |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for device, payload in local_devices.items():
    for method in V4_METHODS:
        metric = payload['methods'][method]
        energy = 'n/a' if metric['dynamic_energy_j'] is None else f"{metric['dynamic_energy_j']:.4f}"
        summary_lines.append(f"| {device.upper()} | {method} | {metric['dice_after']:.4f} | {metric['tre_mm']:.4f} | {energy} | {metric['model_inference_ms']:.1f} | {metric['postprocess_ms']:.1f} | {metric['total_runtime_ms']:.1f} |")
summary_path = OUTPUT_ROOT / 'benchmark_summary_v4_local.md'
summary_path.write_text('\n'.join(summary_lines) + '\n')
print('Wrote', chart_path)
print('Wrote', summary_path)


In [ ]:
# Board-style single-slice V4 preview for the first evaluated pair.
preview_device = torch.device(REQUESTED_DEVICE_NAMES[0])
preview_model = load_board_cpu_reference(V4_WEIGHTS).to(preview_device).eval()
preview_moving_idx, preview_fixed_idx = test_pairs[0]
preview_values = canonical_pair(test_ds, preview_moving_idx, preview_fixed_idx)
preview_raw, preview_canonical = preview_values[:4], preview_values[4:]
preview_moving, preview_fixed, preview_moving_seg, preview_fixed_seg = preview_raw
preview_moving_c, preview_fixed_c, preview_moving_seg_c, preview_fixed_seg_c = preview_canonical
preview_z = preview_moving_c.shape[0] // 2
preview_m_stack, preview_transform = letterbox_stack(v4_extract_stack(preview_moving_c, 'axial', preview_z), 'axial')
preview_f_stack, _ = letterbox_stack(v4_extract_stack(preview_fixed_c, 'axial', preview_z), 'axial')
preview_input = torch.from_numpy(combine_stacks(preview_m_stack, preview_f_stack)[None]).permute(0, 2, 3, 1).to(preview_device)
with torch.no_grad(): preview_flow = preview_model(preview_input)[0].detach().cpu().numpy()
preview_moving_center = preview_m_stack[WINDOW_RADIUS]
preview_fixed_center = preview_f_stack[WINDOW_RADIUS]
preview_warped = spatial_transform_2d(torch.from_numpy(preview_moving_center[None, None]), torch.from_numpy(preview_flow[None]))[0, 0].numpy()
preview_moving_seg_canvas = letterbox_plane(preview_moving_seg_c[preview_z], preview_transform, is_segmentation=True)
preview_fixed_seg_canvas = letterbox_plane(preview_fixed_seg_c[preview_z], preview_transform, is_segmentation=True)
preview_warped_seg = spatial_transform_2d(torch.from_numpy(preview_moving_seg_canvas[None, None].astype(np.float32)), torch.from_numpy(preview_flow[None]), mode='nearest')[0, 0].numpy().astype(np.int16)
preview_before = compute_dice_per_label(preview_moving_seg_canvas, preview_fixed_seg_canvas).mean()
preview_after = compute_dice_per_label(preview_warped_seg, preview_fixed_seg_canvas).mean()
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for axis, image, title in ((axes[0, 0], preview_moving_center, 'Moving V4 canvas'), (axes[0, 1], preview_fixed_center, 'Fixed V4 canvas'), (axes[0, 2], preview_warped, 'Warped moving')):
    axis.imshow(image, cmap='gray'); axis.set_title(title); axis.axis('off')
axes[1, 0].imshow(preview_moving_center, cmap='gray'); add_quiver(axes[1, 0], preview_flow, step=4); axes[1, 0].set_title('Flow'); axes[1, 0].axis('off')
axes[1, 1].imshow(preview_moving_seg_canvas, cmap='tab20'); axes[1, 1].set_title(f'Seg before (Dice {preview_before:.4f})'); axes[1, 1].axis('off')
axes[1, 2].imshow(preview_warped_seg, cmap='tab20'); axes[1, 2].set_title(f'Seg after (Dice {preview_after:.4f})'); axes[1, 2].axis('off')
plt.tight_layout()
preview_path = OUTPUT_ROOT / 'single_slice_v4.png'
plt.savefig(preview_path, dpi=150)
plt.show()
print('Wrote', preview_path)
